#### Import Basics Library and File Unloading

Langkah pertama yang harus kita lakukan adalah melakukan import library yang dibutuhkan untuk pengerjaan project ini dan melakukan pembacaan dataset.

Notes :

- Library yang akan kita gunakan adalah pandas (as pd) dan numpy (as np)
- Dataset yang akan digunakan adalah movie_rating_df.csv

Akses dataset :

movie_rating_df.csv = https://dqlabcdn.xeratic.com/dqlab-dataset/movie_rating_df.csv  

In [ ]:
#import library yang dibutuhkan
import pandas as pd
import numpy as np
#lakukan pembacaan dataset
movie_rating_df = pd.read_csv('https://dqlabcdn.xeratic.com/dqlab-dataset/movie_rating_df.csv') #untuk menyimpan movie_rating_df.csv
movie_rating_df.head()

#### Menampilkan 5 data teratas dan info data

Setelah sebelumnya kita sudah menyimpan dataset pada variabel movie_rating_df, hal selanjutnya yang akan kita lakukan adalah menampilkan lima baris teratas dari dataset tersebut dan menampilkan info mengenai tipe data dan jumlah non-null value dari tiap kolom yang ada pada dataset.

In [ ]:
#tampilkan 5 baris teratas dari movive_rating_df
print(movie_rating_df.head())

#tampilkan info mengenai tipe data dari tiap kolom
print(movie_rating_df.info())

#### Add Actors Dataframe

Dari output yang sudah dihasilkan sebelumnya, kita dapat memperoleh list film dengan beberapa metadata seperti isAdult, runtimeMinutes, dan genres nya

Selanjutnya, kita akan menambahkan metadata lain seperti aktor/aktris yang bermain di film tersebut, kita akan menggunakan dataframe lain kemudian akan melakukan join dengan dataframe movie_rating_df.

Dataset yang akan digunakan adalah actor_name.csv

In [ ]:
#Simpan actor_name.csv pada variable name_df
name_df = pd.read_csv(' https://dqlabcdn.xeratic.com/dqlab-dataset/movie_rating_df.csv')
#Tampilkan 5 baris teratas dari name_df
print(name_df.head())
#Tampilkan informasi mengenai tipe data dari tiap kolom pada name_df
print(name_df.info())

#### Add Directors and Writers Dataframe

Dataframe yang akan ditambahkan selanjutnya adalah dataframe yang berisi directors dan writers dari film.

Dataset yang akan digunakan adalah directors_writers.csv

In [ ]:
#Menyimpan dataset pada variabel director_writers
director_writers = pd.read_csv('https://dqlabcdn.xeratic.com/dqlab-dataset/movie_rating_df.csv')
#Manampilkan 5 baris teratas
print(director_writers.head())
#Menampilkan informasi tipe data
print(director_writers.info())

#### Convert into List

Setelah menampilkan informasi mengenai dataframe directors_writer, dapat dilihat bahwa tidak ada nilai NULL pada dataset tersebut. Hal selanjutnya yang akan kita lakukan adalah mengubah director_name dan writer_name dari string menjadi list

Setelah itu, tampilkan 5 baris teratas dari dataframe director_writers

In [ ]:
pd.set_option('display.max_columns', None)
director_writers = pd.read_csv('https://dqlabcdn.xeratic.com/dqlab-dataset/directors_writers.csv')
#Mengubah director_name menjadi list
director_writers['director_name'] = director_writers['director_name'].apply(lambda row: row.split(','))
director_writers['writer_name'] = director_writers['writer_name'].apply(lambda row: row.split(','))
#Tampilkan 5 data teratas
print(director_writers.head())

#### Update name_df

Kita hanya akan membutuhkan kolom nconst, primaryName, dan knownForTitles pada name_df untuk mencocokkan aktor/aktris ini dengan film yang ada.

In [ ]:
pd.set_option('display.max_columns', None)
name_df = pd.read_csv('https://dqlabcdn.xeratic.com/dqlab-dataset/actor_name.csv')
#Kita hanya akan membutuhkan kolom nconst, primaryName, dan knownForTitles
name_df = name_df[['nconst', 'primaryName', 'knownForTitles']]
#Tampilkan 5 baris teratas dari name_df
print(name_df.head())

#### Movies per Actor

Hal selanjutnya yang ingin kita ketahui adalah mengenai variasi dari jumlah film yang dapat dibintangi oleh seorang aktor.

Tentunya seorang aktor dapat membintangi lebih dari 1 film, bukan? maka akan diperlukan untuk membuat table yang mempunyai relasi 1-1 ke masing-masing title movie tersebut. Kita akan melakukan unnest terhadap table tersebut.

Pekerjaan selanjutnya yang harus kita lakukan adalah :

1. Melakukan pengecekan variasi jumlah film yang dibintangi oleh aktor.

2. Mengubah kolom 'knownForTitles' menjadi list of list.

In [ ]:
#Melakukan pengecekan variasi
print(name_df['knownForTitles'].apply(lambda x: len(x.split(','))).unique())

#Mengubah knownForTitles menjadi list of list
name_df['knownForTitles'] = name_df['knownForTitles'].apply(lambda x: x.split(','))

#Mencetak 5 baris teratas
print(name_df.head())

#### Korespondensi 1 - 1

Karena pada data sebelumnya dapat dilihat bahwa seorang aktor dapat membintangi 1 sampai 4 film, diperlukan untuk membuat table yang mempunyai relasi 1-1 dari aktor ke masing-masing title movie tersebut.




In [ ]:
pd.set_option('display.max_columns', None)
name_df = pd.read_csv('https://dqlabcdn.xeratic.com/dqlab-dataset/actor_name.csv')
name_df = name_df[['nconst','primaryName','knownForTitles']]
name_df['knownForTitles'] = name_df['knownForTitles'].apply(lambda x: x.split(','))
#menyiapkan bucket untuk dataframe
df_uni = []
for x in ['knownForTitles']:
    #mengulang index dari tiap baris sampai tiap elemen dari knownForTitles
    idx = name_df.index.repeat(name_df['knownForTitles'].str.len())
    #memecah values dari list di setiap baris dan menggabungkan nya dengan rows lain menjadi dataframe
    df1 = pd.DataFrame({
        x: np.concatenate(name_df[x].values)
    })
    #mengganti index dataframe tersebut dengan idx yang sudah kita define di awal
    df1.index = idx
    #untuk setiap dataframe yang terbentuk, kita append ke dataframe bucket
    df_uni.append(df1)
#menggabungkan semua dataframe menjadi satu
df_concat = pd.concat(df_uni, axis=1)
#left join dengan value dari dataframe yang awal
unnested_df = df_concat.join(name_df.drop(['knownForTitles'], axis=1), how='left')
#select kolom sesuai dengan dataframe awal
unnested_df = unnested_df[name_df.columns.tolist()]
print(unnested_df)

#### Mengelompokkan primaryName menjadi list group by knownForTitles

Selanjutnya, kita akan melakukan grouping kembali pada kolom player karena yang kita inginkan adalah level movie untuk melakukan movie recommendation

In [ ]:
unnested_drop = unnested_df.drop(['nconst'], axis=1)
#menyiapkan bucket untuk dataframe
df_uni = []
for col in ['primaryName']:
    #agregasi kolom PrimaryName sesuai group_col yang sudah di define di atas
    dfi = unnested_drop.groupby(['knownForTitles'])[col].apply(list)
    #Lakukan append
    df_uni.append(dfi)
df_grouped = pd.concat(df_uni, axis=1).reset_index()
df_grouped.columns = ['knownForTitles','cast_name']
print(df_grouped)

#### Join table

Sekarang kita akan melakukan

- join antara movie table dan cast table (field knownForTitles dan tconst)

- join antara base_df dengan director_writer table (filed tconts dan tconst)

In [ ]:
pd.set_option('display.max_columns', None)

movie_rating_df = pd.read_csv('https://dqlabcdn.xeratic.com/dqlab-dataset/movie_rating_df.csv')

director_writers = pd.read_csv('https://dqlabcdn.xeratic.com/dqlab-dataset/directors_writers.csv')
director_writers['director_name'] = director_writers['director_name'].apply(lambda row: row.split(','))
director_writers['writer_name'] = director_writers['writer_name'].apply(lambda row: row.split(','))

name_df = pd.read_csv('https://dqlabcdn.xeratic.com/dqlab-dataset/actor_name.csv')
name_df = name_df[['nconst','primaryName','knownForTitles']]
name_df['knownForTitles'] = name_df['knownForTitles'].apply(lambda x: x.split(','))

df_uni = []
for x in ['knownForTitles']:
    idx = name_df.index.repeat(name_df['knownForTitles'].str.len())
    df1 = pd.DataFrame({
        x: np.concatenate(name_df[x].values)
    })
    df1.index = idx
    df_uni.append(df1)

df_concat = pd.concat(df_uni, axis=1)
unnested_df = df_concat.join(name_df.drop(columns=['knownForTitles']), how='left')
unnested_df = unnested_df[name_df.columns.tolist()]

unnested_drop = unnested_df.drop(['nconst'], axis=1)
df_uni = []
for col in ['primaryName']:
    dfi = unnested_drop.groupby(['knownForTitles'])[col].apply(list)
    df_uni.append(dfi)
df_grouped = pd.concat(df_uni, axis=1).reset_index()
df_grouped.columns = ['knownForTitles','cast_name']

#join antara movie table (movie_rating_df) dan cast table (df_grouped)
base_df = pd.merge(df_grouped, movie_rating_df, left_on='knownForTitles', right_on='tconst', how='inner')

#join antara base_df dengan director_writer table
base_df = pd.merge(base_df, director_writers, left_on='tconst', right_on='tconst', how='inner')
print(base_df.head())

#### Cleaning data

Setelah melakukan join table sebelumnya, sekarang hal yang akan kembali kita lakukan adalah melakukan cleaning pada data yang sudah dihasilkan.

In [ ]:
#Melakukan drop terhadap kolom knownForTitles
base_drop = base_df.drop(['knownForTitles'], axis=1)
print(base_drop.info())
#Mengganti nilai NULL pada kolom genres dengan 'Unknown'
base_drop['genres'] = base_drop['genres'].fillna('Unknown')
#Melakukan perhitungan jumlah nilai NULL pada tiap kolom
print(base_drop.isnull().sum())
#Mengganti nilai NULL pada kolom dorector_name dan writer_name dengan 'Unknown'
base_drop[['director_name','writer_name']] = base_drop[['director_name','writer_name']].fillna('unknown')
#karena value kolom genres terdapat multiple values, jadi kita akan bungkus menjadi list of list
base_drop['genres'] = base_drop['genres'].apply(lambda x: x.split(','))

#### Reformat table base_df

Hal selanjutnya yang akan kita lakukan adalah melakukan reformat pada table base_df yang beberapa kolomnya sudah didrop.

Petunjuk:

Rename-lah kolom berikut:

- primaryTitle -> title
- titleType -> type
- startYear -> start
- runtimeMinutes -> duration
- averageRating -> rating
- numVotes -> votes


In [ ]:
#Drop kolom tconst, isAdult, endYear, originalTitle
base_drop2 = base_drop.drop(['tconst','isAdult','endYear','originalTitle'], axis=1)
base_drop2 = base_drop2[['primaryTitle','titleType','startYear','runtimeMinutes','genres','averageRating','numVotes','cast_name','director_name','writer_name']]
base_drop2.columns = ['title','type','start','duration','genres','rating','votes','cast_name','director_name','writer_name']
print(base_drop2.head())

#### Klasifikasi Metadata

kita akan klasifikasikan berdasarkan metadata genres, primaryName (cast name), director name, dan writer_name.

In [ ]:
#Klasifikasi berdasar title, cast_name, genres, director_name, dan writer_name
feature_df = base_drop2[['title','cast_name','genres','director_name','writer_name']]
#Tampilkan 5 baris teratas
print(feature_df.head())

#### Pertanyaan 1: Bagaimana cara membuat fungsi untuk strip spaces dari setiap row dan setiap elemennya?

Lengkapilah function sanitize yang digunakan untuk melakukan strip spaces dari setiap row dan setiap elemennya

In [ ]:
def sanitize(x):
    try:
        #kalau cell berisi list
        if isinstance(x,list):
            return [i.replace(' ','').lower() for i in x]
        #kalau cell berisi string
        else:
            return [x.replace(' ','').lower()]
    except:
        print(x)

#Kolom : cast_name, genres, writer_name, director_name
feature_cols = ['cast_name','genres','writer_name','director_name']
#Apply function sanitize
for col in feature_cols:
    feature_df[col] = feature_df[col].apply(sanitize)

#### Pertanyaan 2: Bagaimana cara membuat fungsi untuk membuat metadata soup (menggabungkan semua feature menjadi 1 bagian kalimat) untuk setiap judulnya?

Lengkapi function soup_feature yang digunakan untuk menggabungkan semua feature  menjadi 1 bagian

In [ ]:
#kolom yang digunakan : cast_name, genres, director_name, writer_name
def soup_feature(x):
    return ' '.join(x['cast_name']) + ' ' + ' '.join(x['genres']) + ' ' + ' '.join(x['director_name']) + ' ' + ' '.join(x['writer_name'])
#membuat soup menjadi 1 kolom
feature_df['soup'] = feature_df.apply(soup_feature, axis=1)

#### Pertanyaan 3: Cara menyiapkan CountVectorizer (stop_words = english) dan fit dengan soup yang kita buat di atas

CountVectorizer adalah tipe paling sederhana dari vectorizer. Supaya lebih mudah akan dijelaskan melalui contoh di bawah ini:

bayangkan terdapat 3 text A, B, dan C, dimana text nya adalah
- A: The Sun is a star
- B: My Love is like a red, red rose
- C : Mary had a little lamb

Sekarang kita harus konversi text-text ini menjadi bentuk vector menggunakan CountVectorizer. Langkah-langkahnya adalah: menghitung ukuran dari vocabulary. Vocabulary adalah jumlah dari kata unik yang ada dari text tersebut.

Oleh sebab itu, vocabulary dari set ketiga text tersebut adalah: the, sun, is, a, star, my, love, like, red, rose, mary, had, little, lamb. Secara total, ukuran vocabulary adalah 14.

Tetapi, biasanya kita tidak include stop words (english), seperti as, is, a, the, dan sebagainya karena itu adalah kata yang sudah common sekali.

Dengan mengeliminasi stop words, maka clean size vocabulary kita adalah like, little, lamb, love, mary, red, rose, sun, star (sorted alphabet ascending)
Maka, dengan menggunakan CountVectorizer, maka hasil yang kita dapatkan adalah sebagai berikut:

A : (0,0,0,0,0,0,0,1,1), terdiri atas sun:1, star:1

B : (1,0,0,1,0,2,1,0,0), terdiri atas like:1, love:1, red:2, rose:1

C : (0,1,1,0,1,0,0,0,0), terdiri atas little:1, lamb:1, mary:1

In [ ]:
#import CountVectorizer
from sklearn.feature_extraction.text import CountVectorizer
#definisikan CountVectorizer dan mengubah soup tadi menjadi bentuk vector
count = CountVectorizer(stop_words='english')
count_matrix = count.fit_transform(feature_df['soup'])
print(count)
print(count_matrix.shape)

#### Pertanyaan 4: Cara membuat model similarity antara count matrix

Pada langkah ini, kita akan menghitung score cosine similarity dari setiap pasangan judul (berdasarkan semua kombinasi pasangan yang ada, dengan kata lain kita akan membuat 675 x 675 matrix, dimana cell di kolom i dan j menunjukkan score similarity antara judul i dan j. kita dapat dengan mudah melihat bahwa matrix ini simetris dan setiap elemen pada diagonal adalah 1, karena itu adalah similarity score dengan dirinya sendiri

Cosine Similarity

pada bagian ini, kita akan menggunakan formula cosine similarity untuk membuat model. Score cosine ini sangatlah berguna dan mudah untuk dihitung.

output yang didapat antara range -1 sampai 1. Score yang hampir mencapai 1 artinya kedua entitas tersebut sangatlah mirip sedangkan score yang hampir mencapai -1 artinya kedua entitas tersebut adalah beda

In [ ]:
#Import cosine_similarity
from sklearn.metrics.pairwise import cosine_similarity
#Gunakan cosine_similarity antara count_matrix
cosine_sim = cosine_similarity(count_matrix, count_matrix)
#print hasilnya
print(cosine_sim)

#### Pertanyaan 5: Cara membuat content based recommender system

Task selanjutnya yang harus dilakukan adalah reverse mapping dengan judul sebagai index nya

In [ ]:
indices = pd.Series(feature_df.index, index=feature_df['title']).drop_duplicates()
def content_recommender(title):
    #mendapatkan index dari judul film (title) yang disebutkan
    idx = indices[title]
#menjadikan list dari array similarity cosine sim
    #hint: cosine_sim[idx]
    sim_scores = list(enumerate(cosine_sim[idx]))
#mengurutkan film dari similarity tertinggi ke terendah
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
#untuk mendapatkan list judul dari item kedua sampe ke 11
    sim_scores = sim_scores[1:11]
#mendapatkan index dari judul-judul yang muncul di sim_scores
    movie_indices = [i[0] for i in sim_scores]
#dengan menggunakan iloc, kita bisa panggil balik berdasarkan index dari movie_indices
    return base_df.iloc[movie_indices]
#aplikasikan function di atas
print(content_recommender('The Lion King'))